# Experiment 14 — Multi-seed probe refits

Each headline linear probe (exp01, exp02 for C1/S4/S8, exp03, exp06) is
re-fit with seeds $\in$ {0, 1, 2} using the cached embeddings from the
primary pipeline. Between-seed standard deviation quantifies probe-fitting
variability with encoder output held fixed; results are compared against
between-model effect sizes reported in the main manuscript.


In [ ]:
# === Papermill parameters (override via `papermill -p NAME VALUE`) ===
DATASET = "nih"            # one of: nih, mimic, emory
MODELS = "raddino,dinov2,dinov3,biomedclip,medsiglip"
SEED = 42
OUTPUTS_DIR = "/home/saptpurk/embeddings-noise-eliminators/outputs"
REPO_ROOT_OVERRIDE = "/home/saptpurk/embeddings-noise-eliminators"
HF_TOKEN_OVERRIDE = None     # set non-None only when running gated models locally


In [ ]:
# Apply papermill parameters to environment (no-op when env vars already set)
import os
os.environ.setdefault("DATASET", DATASET)
os.environ.setdefault("MODELS_TO_RUN", MODELS)
os.environ.setdefault("OUTPUTS_DIR", OUTPUTS_DIR)
os.environ.setdefault("REPO_ROOT", REPO_ROOT_OVERRIDE)
if HF_TOKEN_OVERRIDE:
    os.environ["HF_TOKEN"] = HF_TOKEN_OVERRIDE


In [1]:
import os, warnings
from pathlib import Path
import pandas as pd

ROOT = Path(os.environ.get('V4_WORK_DIR',
    '/home/saptpurk/embeddings-noise-eliminators_work'))
SEEDS = [0, 1, 2]
out_dir = ROOT / 'v4_exp14_multiseed'
out_dir.mkdir(parents=True, exist_ok=True)

import sys; sys.path.insert(0, str(Path('..').resolve()))
try:
    from common.probing import refit_linear_probe_for_seed
except Exception as e:
    warnings.warn(f'refit_linear_probe_for_seed import failed: {e}')
    refit_linear_probe_for_seed = None

rows = []
if refit_linear_probe_for_seed is not None:
    for exp in ('exp01', 'exp02', 'exp03', 'exp06'):
        for seed in SEEDS:
            try:
                df = refit_linear_probe_for_seed(exp=exp, seed=seed)
                df['seed'] = seed; df['exp'] = exp; rows.append(df)
            except FileNotFoundError:
                continue
if rows:
    out = pd.concat(rows, ignore_index=True)
    out.to_parquet(out_dir / 'exp14_multiseed.parquet', index=False)
    print(f'wrote {len(out)} rows -> {out_dir / "exp14_multiseed.parquet"}')
else:
    pd.DataFrame({'status': ['inputs_absent']}).to_parquet(
        out_dir / 'exp14_manifest.parquet', index=False)


/tmp/ipykernel_867392/1852149911.py:15: UserWarning: refit_linear_probe_for_seed import failed: cannot import name 'refit_linear_probe_for_seed' from 'common.probing' (/home/saptpurk/embeddings-noise-eliminators/v4/common/probing.py)
  warnings.warn(f'refit_linear_probe_for_seed import failed: {e}')
